# Accurate and Interpretable Clinical Diagnosis with Kolmogorov–Arnold Networks
## Reproducible companion notebook (code + results in one file)

This self-contained notebook regenerates **every table, statistic, and figure** in the paper,
step by step. All code is inline (no package install required) so a reviewer can run it directly.

**Sections (matching the manuscript):**
1. Setup — dependencies, seeds, publication figure settings (Times New Roman, 300 DPI)
2. Datasets — download, preprocessing, dataset table (Table I)
3. KAN implementation and baseline models (Table II config)
4. Predictive performance — 5-fold CV accuracy (Table III, Fig. 1)
5. Statistical comparison — Friedman, average ranks, Wilcoxon (Fig. 2)
6. Intrinsic interpretability — learned spline functions (Fig. 3)
7. Faithfulness — deletion test (Fig. 4)
8. Clinical plausibility — four-method importance agreement (Fig. 5)
9. Summary of results

**Run:** *Kernel → Restart & Run All*. Full run ≈ 10–20 min on CPU (SHAP + CV loops dominate).
All randomness is seeded (`RS = 42`); results are deterministic run-to-run.


## 1. Setup
Install dependencies (run once), then imports, seeds, and publication figure settings.

In [ ]:
# Run once if needed:
# !pip install torch scikit-learn xgboost shap pandas numpy matplotlib scipy

In [ ]:
import os, json, warnings, urllib.request, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.datasets import load_breast_cancer
from scipy.stats import friedmanchisquare, wilcoxon, spearmanr, rankdata
from itertools import combinations
try:
    from xgboost import XGBClassifier; HAS_XGB = True
except Exception:
    HAS_XGB = False
import shap

%matplotlib inline

# ---- reproducibility ----
RS = 42
np.random.seed(RS); torch.manual_seed(RS)
try: torch.use_deterministic_algorithms(True, warn_only=True)
except Exception: pass
DEVICE = torch.device("cpu")

# ---- output dir ----
OUT = "outputs"; FIG = os.path.join(OUT, "figures")
os.makedirs(FIG, exist_ok=True)

# ---- publication figure settings: Times New Roman + 300 DPI ----
# Prefers real Times New Roman (Windows/Mac); falls back to metric-identical Liberation Serif.
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "Nimbus Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "axes.titlesize": 12, "axes.labelsize": 11,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.facecolor": "white",
})
SAVE_DPI = 300
def savefig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".png"), dpi=SAVE_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(os.path.join(FIG, name + ".pdf"), bbox_inches="tight", facecolor="white")  # vector
print("Setup complete. XGBoost available:", HAS_XGB)

## 2. Datasets and preprocessing
Five public clinical datasets are downloaded and preprocessed identically: the final column is the
label; features are coerced to numeric; targets are integer-encoded; Parkinson's identifier column is
removed and Haberman's survival field binarised. Standardisation is done later, per split, on training
data only (to prevent leakage).

In [ ]:
os.makedirs("med_data", exist_ok=True)
def _get(url, path):
    if not os.path.exists(path): urllib.request.urlretrieve(url, path)
    return path

# Breast Cancer (scikit-learn = UCI WDBC)
bc = load_breast_cancer(); _df = pd.DataFrame(bc.data); _df["target"] = bc.target
_df.to_csv("med_data/breast_cancer.csv", index=False)
# Pima
_get("https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv", "med_data/pima.csv")
# Heart (Cleveland)
_get("https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv", "med_data/heart.csv")
_h = pd.read_csv("med_data/heart.csv"); _h.columns=[c.strip().replace("\ufeff","") for c in _h.columns]
_h.to_csv("med_data/heart.csv", index=False)
# Parkinson's (drop name; status -> target)
_get("https://raw.githubusercontent.com/lvwarren/Parkinsons/main/parkinsons.csv", "med_data/_p.csv")
_p = pd.read_csv("med_data/_p.csv"); _st=_p["status"]
_p = _p.drop(columns=[c for c in _p.columns if c.lower() in ("name","status")]); _p["target"]=_st.values
_p.to_csv("med_data/parkinsons.csv", index=False)
# Haberman (binarise)
_get("https://raw.githubusercontent.com/jbrownlee/Datasets/master/haberman.csv", "med_data/_hb.csv")
_hb = pd.read_csv("med_data/_hb.csv", header=None); _hb.columns=["age","year","nodes","status"]
_hb["target"]=(_hb["status"]==2).astype(int); _hb=_hb.drop(columns=["status"])
_hb.to_csv("med_data/haberman.csv", index=False)
print("Datasets ready.")

In [ ]:
BC_FEATS = [n.replace(" ","_")[:12] for n in load_breast_cancer().feature_names]
DATASETS = {
 "Haberman":    ("med_data/haberman.csv",    ["age","year","nodes"]),
 "Pima":        ("med_data/pima.csv",         ["Pregnancies","Glucose","BloodPressure","SkinThickness","Insulin","BMI","Pedigree","Age"]),
 "Heart":       ("med_data/heart.csv",        ["age","sex","cp","trestbps","chol","fbs","restecg","thalach","exang","oldpeak","slope","ca","thal"]),
 "Parkinsons":  ("med_data/parkinsons.csv",   ["Fo","Fhi","Flo","Jitter%","JitterAbs","RAP","PPQ","DDP","Shimmer","ShimmerdB","APQ3","APQ5","APQ","DDA","NHR","HNR","RPDE","DFA","spread1","spread2","D2","PPE"]),
 "BreastCancer":("med_data/breast_cancer.csv",BC_FEATS),
}
ORDER = ["Haberman","Pima","Heart","Parkinsons","BreastCancer"]

def load(name):
    path, feats = DATASETS[name]
    df = pd.read_csv(path)
    X = df.iloc[:,:-1].apply(pd.to_numeric, errors="coerce"); X = X.fillna(X.median()).values.astype(np.float32)
    yc = df.iloc[:,-1]
    y = (LabelEncoder().fit_transform(yc.astype(str)) if yc.dtype==object else yc.values).astype(np.int64)
    return X, y, feats

# ----- Table I: dataset description -----
rows=[]
for n in ORDER:
    X,y,_ = load(n)
    collin = np.mean(np.abs(np.corrcoef(X.T)[np.triu_indices(X.shape[1],1)]))
    vc=np.bincount(y); rows.append([n, X.shape[0], X.shape[1], f"{vc[0]}/{vc[1]}", round(float(collin),3)])
table1 = pd.DataFrame(rows, columns=["Dataset","Samples","Features","Class balance","Mean|corr|"])
table1.to_csv(os.path.join(OUT,"table1_datasets.csv"), index=False)
print("TABLE I — Dataset description"); table1

## 3. KAN implementation and baseline models
The KAN is implemented from scratch: each edge is a learnable univariate function
$\phi(x)=w_b\,\mathrm{SiLU}(x)+w_s\sum_k c_k B_k(x)$ (cubic B-splines on a grid), and nodes sum
their incoming edges. Baselines: logistic regression, RBF-SVM, random forest, XGBoost, and an MLP.

In [ ]:
class KANLinear(nn.Module):
    def __init__(self, in_f, out_f, grid_size=5, spline_order=3, grid_range=(-1,1)):
        super().__init__()
        self.in_f,self.out_f,self.grid_size,self.spline_order = in_f,out_f,grid_size,spline_order
        h=(grid_range[1]-grid_range[0])/grid_size
        grid=torch.arange(-spline_order, grid_size+spline_order+1)*h+grid_range[0]
        self.register_buffer("grid", grid.expand(in_f,-1).contiguous())
        self.base_weight   = nn.Parameter(torch.empty(out_f,in_f))
        self.spline_weight = nn.Parameter(torch.empty(out_f,in_f,grid_size+spline_order))
        self.spline_scaler = nn.Parameter(torch.empty(out_f,in_f))
        self.base_act = nn.SiLU()
        nn.init.kaiming_uniform_(self.base_weight, a=np.sqrt(5))
        nn.init.normal_(self.spline_weight, 0, 0.1)
        nn.init.kaiming_uniform_(self.spline_scaler, a=np.sqrt(5))
    def b_splines(self, x):
        g=self.grid; x=x.unsqueeze(-1)
        b=((x>=g[:,:-1])&(x<g[:,1:])).to(x.dtype)
        for k in range(1,self.spline_order+1):
            b=((x-g[:,:-(k+1)])/(g[:,k:-1]-g[:,:-(k+1)])*b[:,:,:-1])+((g[:,k+1:]-x)/(g[:,k+1:]-g[:,1:-k])*b[:,:,1:])
        return b.contiguous()
    def forward(self, x):
        base=F.linear(self.base_act(x), self.base_weight)
        sp=F.linear(self.b_splines(x).view(x.size(0),-1),
                    (self.spline_weight*self.spline_scaler.unsqueeze(-1)).view(self.out_f,-1))
        return base+sp

class KAN(nn.Module):
    def __init__(self, layers, grid_size=5, spline_order=3):
        super().__init__()
        self.layers=nn.ModuleList([KANLinear(layers[i],layers[i+1],grid_size,spline_order) for i in range(len(layers)-1)])
    def forward(self, x):
        for l in self.layers: x=l(x)
        return x

class MLP(nn.Module):
    def __init__(self, d_in, d_out, dropout=0.2):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(d_in,32),nn.BatchNorm1d(32),nn.ReLU(),nn.Dropout(dropout),
                               nn.Linear(32,16),nn.BatchNorm1d(16),nn.ReLU(),nn.Dropout(dropout),nn.Linear(16,d_out))
    def forward(self,x): return self.net(x)

def sk_models():
    m={"Logistic Regression":lambda:LogisticRegression(max_iter=2000),
       "SVM (RBF)":lambda:SVC(kernel="rbf",C=10,gamma="scale",probability=True),
       "Random Forest":lambda:RandomForestClassifier(n_estimators=300,random_state=RS,n_jobs=-1)}
    if HAS_XGB:
        m["XGBoost"]=lambda:XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.1,
                                          eval_metric="logloss",random_state=RS,n_jobs=-1)
    return m
print("Models defined.")

In [ ]:
def train_torch(model, Xtr,ytr,Xva,yva, epochs=200, patience=25, lr=5e-3, batch=64, seed=RS):
    model=model.to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4); crit=nn.CrossEntropyLoss()
    g=torch.Generator(); g.manual_seed(seed)  # seeded shuffling -> deterministic
    loader=DataLoader(TensorDataset(torch.tensor(Xtr,dtype=torch.float32),torch.tensor(ytr,dtype=torch.long)),
                      batch_size=batch, shuffle=True, generator=g, num_workers=0)
    Xv=torch.tensor(Xva,dtype=torch.float32,device=DEVICE); yv=torch.tensor(yva,dtype=torch.long,device=DEVICE)
    best=np.inf; bs=None; wait=0; ran=0
    for ep in range(epochs):
        ran=ep+1; model.train()
        for xb,yb in loader:
            opt.zero_grad(); crit(model(xb.to(DEVICE)),yb.to(DEVICE)).backward(); opt.step()
        model.eval()
        with torch.no_grad(): vl=crit(model(Xv),yv).item()
        if vl<best-1e-4: best,bs,wait=vl,{k:v.cpu().clone() for k,v in model.state_dict().items()},0
        else:
            wait+=1
            if wait>=patience: break
    if bs: model.load_state_dict(bs)
    model._epochs_ran=ran; return model

def proba(model, Z):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.tensor(np.asarray(Z,dtype=np.float32),device=DEVICE)),1).cpu().numpy()
print("Training utilities defined.")

## 4. Predictive performance — stratified 5-fold cross-validation
Per-fold standardisation (fit on train only) and an inner 15% validation split for early stopping.
Produces **Table III** and **Fig. 1**.

In [ ]:
def auc(y,pp,nc): return roc_auc_score(y,pp[:,1]) if nc==2 else roc_auc_score(y,pp,multi_class="ovr")

def run_cv(name):
    X,y,_ = load(name); d=X.shape[1]; nc=len(np.unique(y))
    skf=StratifiedKFold(5, shuffle=True, random_state=RS); sk=sk_models()
    tb={"MLP":lambda:MLP(d,nc), "KAN":lambda:KAN([d,24,nc])}
    res={k:{"acc":[],"f1":[],"auc":[],"epochs":[]} for k in list(sk)+list(tb)}
    for tr,te in skf.split(X,y):
        sc=StandardScaler().fit(X[tr]); Xtr,Xte=sc.transform(X[tr]),sc.transform(X[te]); ytr,yte=y[tr],y[te]
        Xin,Xv,yin,yv=train_test_split(Xtr,ytr,test_size=0.15,stratify=ytr,random_state=RS)
        for k,c in sk.items():
            m=c().fit(Xtr,ytr); pp=m.predict_proba(Xte); pr=pp.argmax(1)
            res[k]["acc"].append(accuracy_score(yte,pr)); res[k]["f1"].append(f1_score(yte,pr,average="macro")); res[k]["auc"].append(auc(yte,pp,nc))
        for k,c in tb.items():
            torch.manual_seed(RS); m=train_torch(c(),Xin,yin,Xv,yv); pp=proba(m,Xte); pr=pp.argmax(1)
            res[k]["acc"].append(accuracy_score(yte,pr)); res[k]["f1"].append(f1_score(yte,pr,average="macro"))
            res[k]["auc"].append(auc(yte,pp,nc)); res[k]["epochs"].append(m._epochs_ran)
    return {k:{"acc":(float(np.mean(v["acc"])),float(np.std(v["acc"]))),"f1":float(np.mean(v["f1"])),
               "auc":float(np.mean(v["auc"])),"epochs":float(np.mean(v["epochs"])) if v["epochs"] else None,"d":d}
            for k,v in res.items()}

ACC={}
for n in ORDER:
    print("running", n, "..."); ACC[n]=run_cv(n)
json.dump(ACC, open(os.path.join(OUT,"accuracy_results.json"),"w"), indent=2)
MODELS=["Logistic Regression","SVM (RBF)","Random Forest"]+(["XGBoost"] if HAS_XGB else [])+["MLP","KAN"]
print("done")

In [ ]:
# TABLE III — 5-fold accuracy (%)
table3 = pd.DataFrame({m:[round(ACC[d][m]["acc"][0]*100,2) for d in ORDER] for m in MODELS}, index=ORDER).T
table3.to_csv(os.path.join(OUT,"table3_accuracy.csv"))
print("TABLE III — Five-fold cross-validated accuracy (%)"); table3

In [ ]:
# FIGURE 1 — accuracy with error bars
palette={"Logistic Regression":"#999","SVM (RBF)":"#bbb","Random Forest":"#7570b3","XGBoost":"#5e4fa2","MLP":"#888","KAN":"#d95f02"}
fig,ax=plt.subplots(figsize=(10,4.8)); x=np.arange(len(ORDER)); w=0.8/len(MODELS)
for i,m in enumerate(MODELS):
    accs=[ACC[d][m]["acc"][0]*100 for d in ORDER]; errs=[ACC[d][m]["acc"][1]*100 for d in ORDER]
    ax.bar(x+(i-len(MODELS)/2+0.5)*w, accs, w, yerr=errs, capsize=2, label=m, color=palette.get(m,"#777"), edgecolor="black", lw=0.3)
ax.set_xticks(x); ax.set_xticklabels([f"{d}\n(d={ACC[d]['KAN']['d']})" for d in ORDER])
ax.set_ylabel("5-fold accuracy (%)"); ax.set_ylim(60,100); ax.legend(ncol=3); ax.grid(axis="y",alpha=.3)
ax.set_title("Mean five-fold cross-validation accuracy across the five clinical datasets")
savefig(fig,"fig_accuracy"); plt.show()

## 5. Statistical comparison (Demšar)
Rank models per dataset (rank 1 = best), average ranks, and test with **Friedman**; then pairwise
**Wilcoxon** of KAN vs each baseline. Produces **Fig. 2**.

In [ ]:
A=np.array([[ACC[d][m]["acc"][0] for m in MODELS] for d in ORDER])   # datasets x models
chi2,p = friedmanchisquare(*[A[:,j] for j in range(len(MODELS))])
ranks=np.vstack([rankdata(-A[i]) for i in range(len(ORDER))]); avg=ranks.mean(0)
ki=MODELS.index("KAN")
wil={m: round(float(wilcoxon(A[:,ki],A[:,j],zero_method="pratt")[1]),3) for j,m in enumerate(MODELS) if j!=ki}
sig={"friedman_chi2":round(float(chi2),3),"friedman_p":round(float(p),4),
     "avg_ranks":{m:round(float(a),2) for m,a in zip(MODELS,avg)},"wilcoxon_kan_vs":wil}
json.dump(sig, open(os.path.join(OUT,"significance.json"),"w"), indent=2)
print(f"Friedman: chi^2 = {chi2:.3f}, p = {p:.4f}  (p>0.05 -> no significant difference)\n")
print("Average ranks (lower = better):")
for m,a in sorted(zip(MODELS,avg), key=lambda t:t[1]): print(f"   {m:<20} {a:.2f}")
print("\nWilcoxon KAN vs each (p):", wil)

In [ ]:
# FIGURE 2 — average ranks
items=sorted(zip(MODELS,avg), key=lambda t:t[1]); names=[m for m,_ in items]; vals=[v for _,v in items]
fig,ax=plt.subplots(figsize=(7,3.6)); yp=list(range(len(names)))[::-1]
ax.barh(yp, vals, color=["#d95f02" if n=="KAN" else "#7570b3" for n in names], edgecolor="black")
ax.set_yticks(yp); ax.set_yticklabels(names)
for i,v in zip(yp,vals): ax.text(v+0.03,i,f"{v:.2f}",va="center",fontsize=9)
ax.set_xlabel("Average rank across 5 datasets (lower = better)"); ax.set_xlim(0,len(MODELS))
ax.set_title(f"Average model rank; Friedman $p$ = {p:.3f}")
savefig(fig,"fig_ranks"); plt.show()

## 6. Intrinsic interpretability — learned spline functions
Train a KAN on Pima and plot the learned edge functions for the four most influential diabetes
features. Produces **Fig. 3**.

In [ ]:
def fit_kan(name, layers_hidden=24, seed=RS):
    X,y,feats=load(name); d=X.shape[1]
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,stratify=y,random_state=RS)
    sc=StandardScaler().fit(Xtr); Xtr=sc.transform(Xtr); Xte=sc.transform(Xte)
    Xin,Xv,yin,yv=train_test_split(Xtr,ytr,test_size=0.15,stratify=ytr,random_state=RS)
    torch.manual_seed(seed); kan=train_torch(KAN([d,layers_hidden,2]),Xin,yin,Xv,yv)
    return kan,d,feats,Xtr,ytr,Xte,yte

kan_p,dp,feats_p,Xtr_p,ytr_p,Xte_p,yte_p = fit_kan("Pima")

# FIGURE 3 — learned splines for Glucose, BMI, Pregnancies, Age
l0=kan_p.layers[0]; xs=torch.linspace(-2.5,2.5,200); top=[1,5,0,7]
fig,axes=plt.subplots(1,4,figsize=(15,3.4))
for k,fi in enumerate(top):
    xin=torch.zeros(len(xs),dp); xin[:,fi]=xs
    with torch.no_grad():
        bs=l0.b_splines(xin)[:,fi,:]; base=l0.base_act(xs).unsqueeze(1)*l0.base_weight[:,fi]
        ssw=(l0.spline_weight*l0.spline_scaler.unsqueeze(-1))[:,fi,:]; Y=(base+bs@ssw.T).numpy()
    for j in range(Y.shape[1]): axes[k].plot(xs.numpy(),Y[:,j],color="#d95f02",alpha=.2,lw=.7)
    axes[k].plot(xs.numpy(),Y.mean(1),color="black",lw=2); axes[k].axhline(0,color="gray",lw=.5)
    axes[k].set_title(feats_p[fi]); axes[k].set_xlabel("standardised value")
    if k==0: axes[k].set_ylabel(r"learned $\phi(x)$")
fig.suptitle("KAN learned spline (edge) functions for the four most influential diabetes features", y=1.04)
savefig(fig,"splines_Pima"); plt.show()

## 7. Faithfulness — deletion test
Define the four attribution methods, then ablate features in importance order (set to standardised
mean 0) and record accuracy; a faithful ranking degrades accuracy faster than random. Produces **Fig. 4**.

In [ ]:
def spline_importance(kan,d):
    l0=kan.layers[0]; xs=torch.linspace(-2.5,2.5,100); imp=np.zeros(d)
    for fi in range(d):
        xin=torch.zeros(len(xs),d); xin[:,fi]=xs
        with torch.no_grad():
            bs=l0.b_splines(xin)[:,fi,:]; base=l0.base_act(xs).unsqueeze(1)*l0.base_weight[:,fi]
            ssw=(l0.spline_weight*l0.spline_scaler.unsqueeze(-1))[:,fi,:]; y=(base+bs@ssw.T).mean(1).numpy()
        imp[fi]=y.max()-y.min()
    return imp
def perm_importance(kan,X,y,rep=10):
    base=accuracy_score(y,proba(kan,X).argmax(1)); imp=np.zeros(X.shape[1]); rng=np.random.default_rng(RS)
    for j in range(X.shape[1]):
        dd=[]
        for _ in range(rep):
            Xp=X.copy(); rng.shuffle(Xp[:,j]); dd.append(base-accuracy_score(y,proba(kan,Xp).argmax(1)))
        imp[j]=np.mean(dd)
    return imp
def shap_importance(kan,Xtr,Xte,n=80):
    bg=shap.kmeans(Xtr,15); ex=shap.KernelExplainer(lambda Z: proba(kan,Z)[:,1], bg)
    return np.abs(ex.shap_values(Xte[:min(n,len(Xte))], nsamples=100, silent=True)).mean(0)
def deletion(kan,Xte,yte,order):
    Xa=Xte.copy(); accs=[accuracy_score(yte,proba(kan,Xa).argmax(1))]
    for f in order: Xa[:,f]=0.0; accs.append(accuracy_score(yte,proba(kan,Xa).argmax(1)))
    return float(np.mean(accs))
def nz(a): a=np.asarray(a,float); return (a-a.min())/(a.max()-a.min()+1e-9)

# fit KAN + all four importances on every dataset (reused in Section 8 too)
XAI={}
for n in ORDER:
    kan,d,feats,Xtr,ytr,Xte,yte = fit_kan(n)
    imps={
        "Spline":      spline_importance(kan,d),
        "Permutation": perm_importance(kan,Xte,yte),
        "SHAP":        shap_importance(kan,Xtr,Xte),
        "RF":          RandomForestClassifier(n_estimators=300,random_state=RS).fit(Xtr,ytr).feature_importances_,
    }
    XAI[n]=dict(kan=kan,d=d,feats=feats,Xte=Xte,yte=yte,imps=imps)
    print("explained", n)
print("done")

In [ ]:
# FIGURE 4 — deletion-based faithfulness (all five datasets)
FAITH={}
for n in ORDER:
    r=XAI[n]; d=r["d"]
    cons=nz(r["imps"]["Spline"])+nz(r["imps"]["Permutation"])+nz(r["imps"]["SHAP"])+nz(r["imps"]["RF"])
    rng=np.random.default_rng(RS)
    rnd=np.mean([deletion(r["kan"],r["Xte"],r["yte"],list(rng.permutation(d))) for _ in range(5)])
    fc=deletion(r["kan"],r["Xte"],r["yte"],list(np.argsort(-cons)))
    FAITH[n]={"consensus":fc,"random":float(rnd),"d":d}
json.dump(FAITH, open(os.path.join(OUT,"faithfulness.json"),"w"), indent=2)

fig,ax=plt.subplots(figsize=(8,4.3)); x=np.arange(len(ORDER)); w=0.38
ax.bar(x-w/2,[FAITH[d]["consensus"] for d in ORDER],w,label="Importance-ordered ablation",color="#1b9e77",edgecolor="black")
ax.bar(x+w/2,[FAITH[d]["random"] for d in ORDER],w,label="Random ablation",color="#999",edgecolor="black",hatch="//")
for i,d in enumerate(ORDER):
    better=FAITH[d]["consensus"]<FAITH[d]["random"]-0.005
    ax.annotate("\u2713" if better else "\u2248",(i,max(FAITH[d]["consensus"],FAITH[d]["random"])+0.008),
                ha="center",fontsize=12,color="#1b9e77" if better else "#999",weight="bold")
ax.set_xticks(x); ax.set_xticklabels([f"{d}\n(d={FAITH[d]['d']})" for d in ORDER],fontsize=8)
ax.set_ylabel("Mean accuracy after feature deletion\n(lower = more faithful)"); ax.legend(); ax.set_ylim(0.6,0.95); ax.grid(axis="y",alpha=.3)
ax.set_title("Deletion-based faithfulness across the five datasets")
savefig(fig,"fig_faith_all"); plt.show()
print({d:(round(FAITH[d]['consensus'],3),round(FAITH[d]['random'],3)) for d in ORDER})

## 8. Clinical plausibility — four-method importance agreement
For each dataset, plot the four normalised importances and report mean pairwise Spearman $\rho$.
Produces **Fig. 5** and exports per-method top features (used to check the §IV-D claims).

In [ ]:
def rho_of(imps): 
    return float(np.nanmean([spearmanr(imps[a],imps[b]).correlation for a,b in combinations(imps,2)]))

export={}
fig,axes=plt.subplots(3,2, figsize=(14,12)); axes=axes.ravel()
panel="abcde"
for idx,n in enumerate(ORDER):
    r=XAI[n]; feats=r["feats"]; d=r["d"]; imps=r["imps"]; rho=rho_of(imps)
    order_idx=np.argsort(imps["Spline"])[::-1]; xq=np.arange(d); w=0.2; ax=axes[idx]
    for off,(lab,col,key) in zip([-1.5,-0.5,0.5,1.5],
        [("KAN spline","#d95f02","Spline"),("Permutation","#1b9e77","Permutation"),("SHAP","#7570b3","SHAP"),("RF (external)","#999","RF")]):
        ax.bar(xq+off*w, nz(imps[key])[order_idx], w, label=lab, color=col)
    ax.set_xticks(xq); ax.set_xticklabels([feats[i] for i in order_idx], rotation=45, ha="right", fontsize=6)
    ax.set_ylabel("normalised importance"); ax.set_title(f"({panel[idx]}) {n}  (mean $\\rho$ = {rho:.2f})", fontsize=10)
    if idx==0: ax.legend(fontsize=7, ncol=2)
    export[n]={"mean_rho":round(rho,4),
               "top5_per_method":{k:[feats[i] for i in np.argsort(-np.asarray(imps[k]))[:5]] for k in imps}}
axes[-1].axis("off")
fig.suptitle("Normalized feature-importance rankings: KAN spline, SHAP, permutation, Random Forest", y=1.01, fontsize=13)
savefig(fig,"imp_all"); plt.show()
json.dump(export, open(os.path.join(OUT,"explainability.json"),"w"), indent=2)
print("\nMean rho per dataset:", {n:export[n]["mean_rho"] for n in ORDER})
print("\nTop features (glucose should lead Pima):")
for n in ORDER: print(f"  {n}: KAN spline top5 = {export[n]['top5_per_method']['Spline'][:5]}")

## 9. Summary of results

In [ ]:
print("="*60); print("SUMMARY"); print("="*60)
print(f"Friedman p = {sig['friedman_p']}  (no significant difference among models)")
ranked=sorted(sig['avg_ranks'].items(), key=lambda t:t[1])
print("KAN average-rank position:", [m for m,_ in ranked].index("KAN")+1, "of", len(MODELS))
nbeat=sum(1 for d in ORDER if FAITH[d]['consensus']<FAITH[d]['random']-0.005)
print(f"KAN explanations beat random ablation on {nbeat}/5 datasets")
print("Explanation agreement (rho):", {n:export[n]['mean_rho'] for n in ORDER})
print("\nArtifacts written to ./outputs/ :")
for f in sorted(os.listdir(OUT)):
    if os.path.isfile(os.path.join(OUT,f)): print("   ", f)
print("Figures (300-DPI PNG + vector PDF) in ./outputs/figures/")

---
*This notebook reproduces the manuscript's tables and figures end-to-end. Neural-model accuracies may
differ by ~1–2 points across PyTorch versions/hardware; the statistical conclusions (Friedman
non-significance, faithfulness 4/5, clinical plausibility) are stable. All figures use Times New Roman
and are exported at 300 DPI for direct inclusion in the paper.*